# RAID v2 end-to-end Colab runner

This notebook mounts Google Drive, clones the `jp_branch` branch, installs dependencies, downloads the datasets needed for Phases A-E, and then runs the matrix phase by phase. All outputs are written under `RAID_ARTIFACT_ROOT` on Drive so the run is resumable after Colab disconnects.

In [1]:
# === 1. Mount Drive and set artifact root ===
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

os.environ['RAID_ARTIFACT_ROOT'] = '/content/drive/MyDrive/raid_v2'
ARTIFACT_ROOT = Path(os.environ['RAID_ARTIFACT_ROOT'])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('RAID_ARTIFACT_ROOT =', ARTIFACT_ROOT)

Mounted at /content/drive
RAID_ARTIFACT_ROOT = /content/drive/MyDrive/raid_v2


In [2]:
# === 2. Clone or refresh the repo ===
REPO_URL = 'https://github.com/ConstantinVictorBeatErtel/RAID.git'
REPO_DIR = Path('/content/RAID')
BRANCH = 'jp_branch'

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd /content/RAID
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}

%cd /content/RAID
!git status --short --branch
!ls v2 | head

Cloning into '/content/RAID'...
remote: Enumerating objects: 248, done.
remote: Counting objects: 100% (248/248), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 248 (delta 109), reused 221 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (248/248), 10.73 MiB | 21.25 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/RAID
## jp_branch...origin/jp_branch
configs
conftest.py
datasets
evaluate.py
features.py
heads
__init__.py
legacy
notebooks
README.md


In [3]:
# === 3. Install Python dependencies ===
%cd /content/RAID
!pip install -q -r /content/RAID/v2/requirements.txt

/content/RAID


In [4]:
# === 4. Inline RoboMimic downloader ===
# Mirrors the Stanford CDN into the exact Drive layout v2 expects.
#
# Files are written as:
#   <artifact_root>/data/robomimic/v1.5/<task>/<variant>/<modality>_v141.hdf5
#
# This matches the Colab flow that produced the working Phase A/B/C runs.

import sys
import time
import urllib.request

ROBOMIMIC_BASE = 'http://downloads.cs.stanford.edu/downloads/rt_benchmark'
ROBOMIMIC_ROOT = ARTIFACT_ROOT / 'data' / 'robomimic' / 'v1.5'

def _progress(block_num, block_size, total_size):
    if total_size <= 0:
        return
    pct = min(100, 100 * block_num * block_size / total_size)
    sys.stdout.write(f'\r  {pct:5.1f}%  ({block_num * block_size / 1e6:.1f} / {total_size / 1e6:.1f} MB)')
    sys.stdout.flush()

def fetch_robomimic(task: str, variant: str, modality: str = 'low_dim') -> Path:
    """Download one RoboMimic HDF5 to Drive at the layout v2 expects.

    Files are stored as <task>/<variant>/<modality>_v141.hdf5 so the
    v2 hdf5_path_for resolver finds them on the first candidate path.
    """
    src_name = 'low_dim.hdf5' if modality == 'low_dim' else 'image.hdf5'
    dst = ROBOMIMIC_ROOT / task / variant / f'{modality}_v141.hdf5'
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 100_000:
        print(f'[skip ] {task}/{variant}/{modality}  ({dst.stat().st_size / 1e6:.1f} MB)')
        return dst
    url = f'{ROBOMIMIC_BASE}/{task}/{variant}/{src_name}'
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    print(f'[fetch] {url}')
    t0 = time.time()
    try:
        urllib.request.urlretrieve(url, tmp, reporthook=_progress)
        sys.stdout.write('\n')
        tmp.replace(dst)
    except Exception as exc:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(f'failed to download {url}: {exc}')
    print(f'[done ] {dst}  ({dst.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.1f}s)')
    return dst

# Phase A only needs lift/ph low_dim.
# Phase B uses the low_dim files below.
# Phase C/D use square/ph image.
for task, variant, modality in [
    ('lift', 'ph', 'low_dim'),
    ('lift', 'mh', 'low_dim'),
    ('can', 'ph', 'low_dim'),
    ('can', 'mh', 'low_dim'),
    ('square', 'ph', 'low_dim'),
    ('square', 'mh', 'low_dim'),
    ('transport', 'ph', 'low_dim'),
    ('transport', 'mh', 'low_dim'),
    ('square', 'ph', 'image'),
]:
    fetch_robomimic(task, variant, modality)

[skip ] lift/ph/low_dim  (18.5 MB)
[skip ] lift/mh/low_dim  (47.9 MB)
[skip ] can/ph/low_dim  (45.5 MB)
[skip ] can/mh/low_dim  (112.5 MB)
[skip ] square/ph/low_dim  (50.0 MB)
[skip ] square/mh/low_dim  (123.9 MB)
[skip ] transport/ph/low_dim  (309.0 MB)
[skip ] transport/mh/low_dim  (637.0 MB)
[skip ] square/ph/image  (2603.4 MB)


In [5]:
# === 5. Verify the files are really on Drive before training ===
from pathlib import Path

root = Path('/content/drive/MyDrive/raid_v2/data/robomimic/v1.5')
checks = [
    root / 'lift/ph/low_dim_v141.hdf5',
    root / 'lift/mh/low_dim_v141.hdf5',
    root / 'can/ph/low_dim_v141.hdf5',
    root / 'can/mh/low_dim_v141.hdf5',
    root / 'square/ph/low_dim_v141.hdf5',
    root / 'square/mh/low_dim_v141.hdf5',
    root / 'transport/ph/low_dim_v141.hdf5',
    root / 'transport/mh/low_dim_v141.hdf5',
    root / 'square/ph/image_v141.hdf5',
]

missing = []
for p in checks:
    ok = p.exists()
    print(p, ok, f'{p.stat().st_size/1e6:.1f} MB' if ok else 'MISSING')
    if not ok:
        missing.append(str(p))

if missing:
    raise RuntimeError('Missing RoboMimic files on Drive:\n' + '\n'.join(missing))


/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/lift/ph/low_dim_v141.hdf5 True 18.5 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/lift/mh/low_dim_v141.hdf5 True 47.9 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/can/ph/low_dim_v141.hdf5 True 45.5 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/can/mh/low_dim_v141.hdf5 True 112.5 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/square/ph/low_dim_v141.hdf5 True 50.0 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/square/mh/low_dim_v141.hdf5 True 123.9 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/transport/ph/low_dim_v141.hdf5 True 309.0 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/transport/mh/low_dim_v141.hdf5 True 637.0 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/square/ph/image_v141.hdf5 True 2603.4 MB


In [6]:
# === 6. GPU sanity check ===
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

GPU: True NVIDIA L4


## Phase A

Reproduce the low-dim Lift baseline in the new harness.

In [7]:
# === 6. Run Phase A ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase A

/content/RAID
[run_matrix] expanded 36 cells (phase filter: A)
[run_matrix] [1/36] SKIP completed run_id=15df78d75b0ee339
[run_matrix] [2/36] SKIP completed run_id=377ba3088a0d2afa
[run_matrix] [3/36] SKIP completed run_id=09bf871c18233c17
[run_matrix] [4/36] SKIP completed run_id=ad6409dd573c0def
[run_matrix] [5/36] SKIP completed run_id=a2f505b7852a37a4
[run_matrix] [6/36] SKIP completed run_id=aeb2e5ca987bb964
[run_matrix] [7/36] SKIP completed run_id=7907b99cf3b1681c
[run_matrix] [8/36] SKIP completed run_id=c01ca7007b7764fc
[run_matrix] [9/36] SKIP completed run_id=29eac130d670c4c1
[run_matrix] [10/36] SKIP completed run_id=9081046a4d799a4e
[run_matrix] [11/36] SKIP completed run_id=5add2f1c6cc8527b
[run_matrix] [12/36] SKIP completed run_id=64ebb02f21e2f744
[run_matrix] [13/36] SKIP completed run_id=c06bd23c4cec9148
[run_matrix] [14/36] SKIP completed run_id=b7f40b00eb831070
[run_matrix] [15/36] SKIP completed run_id=22e874d8ff69e3f1
[run_matrix] [16/36] SKIP completed run_id=95c

## Phase B

Cross-dataset low-dim runs on the mixed RoboMimic tasks.

In [8]:
# === 7. Refresh repo before later phases ===
%cd /content/RAID
!git fetch origin
!git checkout jp_branch
!git reset --hard origin/jp_branch

/content/RAID
Already on 'jp_branch'
Your branch is up to date with 'origin/jp_branch'.
HEAD is now at 6868e42 v2: cross-attention RAID + stride + diffusion fix + low-data n_demos


In [9]:
# === 8. Run Phase B ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase B

/content/RAID
[run_matrix] expanded 18 cells (phase filter: B)
[run_matrix] [1/18] SKIP completed run_id=f06379bf716fd7a7
[run_matrix] [2/18] SKIP completed run_id=80acd1e41b5b85d3
[run_matrix] [3/18] SKIP completed run_id=fab815f0f2c9c6ca
[run_matrix] [4/18] SKIP completed run_id=13df8e9f375ab0bc
[run_matrix] [5/18] SKIP completed run_id=98fd4a0b3b009c80
[run_matrix] [6/18] SKIP completed run_id=1f34b6f0cacff221
[run_matrix] [7/18] SKIP completed run_id=38dd273a04bd7f1c
[run_matrix] [8/18] SKIP completed run_id=fad572795e2076a8
[run_matrix] [9/18] SKIP completed run_id=3d347d9236d7607f
[run_matrix] [10/18] SKIP completed run_id=a2e6e8c13507099c
[run_matrix] [11/18] SKIP completed run_id=5dea093d8d8b48ba
[run_matrix] [12/18] SKIP completed run_id=b045b66f45792875
[run_matrix] [13/18] SKIP completed run_id=2b5fc88800585a8a
[run_matrix] [14/18] SKIP completed run_id=8b8df20387dd74e4
[run_matrix] [15/18] SKIP completed run_id=6c357ffa9a4b2783
[run_matrix] [16/18] SKIP completed run_id=e24

## Phase C

Image-feature IDM. DINOv2 CLS features cached once on Drive, then all heads (direct_mlp, raid, raid_xattn, transformer, diffusion) trained at n_demos in {10, 25, 50, 200}. The new `raid_xattn` head replaces RAID's mean-pool retrieval prior with attention weighting. Phase Cs is the same setup but with a 5-step temporal stride (`(o_t, o_{t+5})`) so the image-IDM problem has a meaningful visual delta at 20 Hz.


In [10]:
# === 9. Extract DINOv2 features and run Phase C + Phase Cs ===
%cd /content/RAID
!python3 -m v2.features --encoders dinov2 --datasets robomimic_square_ph_image
!python3 -m v2.run_matrix --phase C
!python3 -m v2.run_matrix --phase Cs


/content/RAID
config.json: 100% 548/548 [00:00<00:00, 2.53MB/s]
model.safetensors: 100% 346M/346M [00:02<00:00, 122MB/s] 
Loading weights: 100% 223/223 [00:00<00:00, 1065.96it/s, Materializing param=layernorm.weight]                                
preprocessor_config.json: 100% 436/436 [00:00<00:00, 2.48MB/s]
The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[features] SKIP /content/drive/MyDrive/raid_v2/features/robomimic_square_ph_image_dinov2_cls.safetensors (already exists, 46.3 MB)
[run_matrix] expanded 60 cells (phase filter: C)
[run_matrix] [1/60] START phase=C head=direct_mlp dataset=robomimic_square_ph_image n_demos=10 seed=42 action_norm=q01_q99 run_id=8c06aa9bcd76792f
[wandb_resume] no WANDB_API_KEY found; running in 

## Phase D

Encoder ablation: re-extract image features with **SigLIP** (semantic, language-aligned) and compare against the DINOv2 (spatial, self-supervised) results from Phase C using the same Transformer head. SigLIP loads via standard `transformers` without any monkey-patching.


In [11]:
# === 10. Extract SigLIP features (encoder ablation) and run Phase D ===
%cd /content/RAID
!python3 -m v2.features --encoders siglip --datasets robomimic_square_ph_image
!python3 -m v2.run_matrix --phase D


/content/RAID
config.json: 100% 432/432 [00:00<00:00, 2.33MB/s]
model.safetensors: 100% 813M/813M [00:03<00:00, 252MB/s] 
Loading weights: 100% 408/408 [00:00<00:00, 843.36it/s, Materializing param=vision_model.post_layernorm.weight]                      
preprocessor_config.json: 100% 368/368 [00:00<00:00, 1.71MB/s]
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[features] SKIP /content/drive/MyDrive/raid_v2/features/robomimic_square_ph_image_siglip_cls.safetensors (already exists, 46.3 MB)
[run_matrix] expanded 3 cells (phase filter: D)
[run_matrix] [1/3] SKIP completed run_id=9bf40e7f25a24aa4
[run_matrix] [2/3] SKIP completed run_id=8db0d7128953939a
[run_matrix] [3/3] SKIP completed run_id=748a255ad372d310
[run_matrix] F

## Phase E

Multi-task retrieval test on **LIBERO-Spatial** (10 tasks pooled). Expects the canonical LIBERO HDF5 release staged at `<RAID_ARTIFACT_ROOT>/data/libero/libero_spatial/*.hdf5`. The cell below verifies the staging, extracts DINOv2 features once (stride-aware caching), and runs the 18-cell matrix at stride=5 — the same temporal scale where image IDM became a real learning problem on RoboMimic Square.


In [ ]:
# === 11. Verify LIBERO data on Drive, cache features, run Phase E ===
%cd /content/RAID

from v2.runtime.drive import data_root
from v2.datasets.libero import find_libero_episodes, LIBERO_SUITES
from pathlib import Path

lroot = data_root() / 'libero'
available = {}
for suite in LIBERO_SUITES:
    suite_dir = lroot / suite
    if not suite_dir.is_dir():
        print(f'[libero] {suite}: not staged (skipped)')
        continue
    try:
        eps = find_libero_episodes(suite, lroot)
    except FileNotFoundError as exc:
        print(f'[libero] {suite}: {exc}')
        continue
    available[suite] = len(eps)
    print(f'[libero] {suite}: {len(eps)} episodes')

if 'libero_spatial' not in available:
    print('Phase E needs libero_spatial staged at /content/drive/MyDrive/raid_v2/data/libero/libero_spatial/*.hdf5')
    print('Skipping phase E.')
else:
    !python3 -m v2.features --encoders dinov2 --datasets libero_spatial
    !python3 -m v2.run_matrix --phase E


## Aggregate results

Build the forest plot from all completed runs. This script must be run as a module.

In [13]:
# === 12. Aggregate completed runs into summary figures ===
%cd /content/RAID
!python3 -m v2.notebooks.03_matrix_results

/content/RAID
[03_matrix_results] wrote /content/drive/MyDrive/raid_v2/results/figures/matrix_forest.png


In [14]:
# === 13. Inspect generated figure paths ===
from pathlib import Path

fig_root = Path('/content/drive/MyDrive/raid_v2/results/figures')
for path in [fig_root / 'matrix_forest.png']:
    print(path, path.exists(), f'{path.stat().st_size / 1e6:.2f} MB' if path.exists() else '')

pred_root = fig_root / 'predictions'
if pred_root.exists():
    grids = sorted(pred_root.glob('*/grid.png'))
    print('prediction grids:', len(grids))
    for p in grids[-10:]:
        print(' ', p)

/content/drive/MyDrive/raid_v2/results/figures/matrix_forest.png True 0.25 MB
prediction grids: 141
  /content/drive/MyDrive/raid_v2/results/figures/predictions/e9a43588aa1cf12f/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/e9fb5ddfea322c8e/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/ea3176cf07c597e6/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/efefa67e893c729e/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/f06379bf716fd7a7/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/f144f8f58f625fb4/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/f1831edcab5bf786/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/f751c7cdc95c78e7/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/fab815f0f2c9c6ca/grid.png
  /content/drive/MyDrive/raid_v2/results/figures/predictions/fad572795e2076a8/grid.png


In [15]:
# === 14. Optional: dry-run resume checks ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase A --dry-run
!python3 -m v2.run_matrix --phase B --dry-run
!python3 -m v2.run_matrix --phase C --dry-run
!python3 -m v2.run_matrix --phase D --dry-run
!python3 -m v2.run_matrix --phase E --dry-run

/content/RAID
[run_matrix] expanded 36 cells (phase filter: A)
{"run_id": "15df78d75b0ee339", "phase": "A", "head": "direct_mlp", "dataset": "robomimic_lift_ph_low_dim", "n_demos": 25, "seed": 42, "encoder": null, "action_norm_mode": "zscore", "completed": true}
{"run_id": "377ba3088a0d2afa", "phase": "A", "head": "direct_mlp", "dataset": "robomimic_lift_ph_low_dim", "n_demos": 25, "seed": 1337, "encoder": null, "action_norm_mode": "zscore", "completed": true}
{"run_id": "09bf871c18233c17", "phase": "A", "head": "direct_mlp", "dataset": "robomimic_lift_ph_low_dim", "n_demos": 25, "seed": 2024, "encoder": null, "action_norm_mode": "zscore", "completed": true}
{"run_id": "ad6409dd573c0def", "phase": "A", "head": "direct_mlp", "dataset": "robomimic_lift_ph_low_dim", "n_demos": 50, "seed": 42, "encoder": null, "action_norm_mode": "zscore", "completed": true}
{"run_id": "a2f505b7852a37a4", "phase": "A", "head": "direct_mlp", "dataset": "robomimic_lift_ph_low_dim", "n_demos": 50, "seed": 133

In [16]:
# === Aggregate matrix results into a clean table ===
import json
from pathlib import Path
from collections import defaultdict
import statistics

ROOT = Path("/content/drive/MyDrive/raid_v2")
runs = ROOT / "runs"

rows = []
for d in sorted(runs.iterdir()):
    rj = d / "result.json"
    mj = d / "metrics.json"
    if not rj.is_file():
        continue
    r = json.loads(rj.read_text())
    cfg = r.get("config", {})
    row = {
        "run_id": r.get("run_id", d.name),
        "phase": cfg.get("phase"),
        "head": cfg.get("head"),
        "dataset": cfg.get("dataset"),
        "encoder": cfg.get("encoder") or "-",
        "n_demos": cfg.get("n_demos"),
        "seed": cfg.get("seed"),
        "best_val_mse": r.get("best_val_mse"),
        "best_epoch": r.get("best_epoch"),
    }
    if mj.is_file():
        m = json.loads(mj.read_text())
        row["eval_val_mse"] = m.get("val_mse")
        row["contact_mse"] = m.get("contact_mse")
    rows.append(row)

# Group by (phase, head, dataset, encoder, n_demos) → mean ± std across seeds
groups = defaultdict(list)
for r in rows:
    key = (r["phase"], r["head"], r["dataset"], r["encoder"], r["n_demos"])
    groups[key].append(r["best_val_mse"])

print(f"\n{'Ph':<3} {'Head':<12} {'Dataset':<32} {'Enc':<8} {'N':>4} | "
      f"{'mean':>8}  {'std':>7}  {'n':>2}")
print("-" * 90)
for key in sorted(groups):
    vals = [v for v in groups[key] if v is not None]
    if not vals:
        continue
    mean = statistics.mean(vals)
    std = statistics.stdev(vals) if len(vals) > 1 else 0.0
    print(f"{key[0]:<3} {str(key[1]):<12} {str(key[2])[:32]:<32} {str(key[3]):<8} "
          f"{str(key[4]):>4} | {mean:>8.4f}  {std:>7.4f}  {len(vals):>2}")

print(f"\nTotal cells: {len(rows)} runs across {len(groups)} configurations")


Ph  Head         Dataset                          Enc         N |     mean      std   n
------------------------------------------------------------------------------------------
A   direct_mlp   robomimic_lift_ph_low_dim        -          25 |   0.3408   0.0114   3
A   direct_mlp   robomimic_lift_ph_low_dim        -          50 |   0.3566   0.0042   3
A   direct_mlp   robomimic_lift_ph_low_dim        -         100 |   0.2981   0.0026   3
A   direct_mlp   robomimic_lift_ph_low_dim        -         200 |   0.1789   0.0043   3
A   knn          robomimic_lift_ph_low_dim        -          25 |   0.6168   0.0000   3
A   knn          robomimic_lift_ph_low_dim        -          50 |   0.7441   0.0000   3
A   knn          robomimic_lift_ph_low_dim        -         100 |   0.7167   0.0000   3
A   knn          robomimic_lift_ph_low_dim        -         200 |   0.5665   0.0000   3
A   raid         robomimic_lift_ph_low_dim        -          25 |   0.4025   0.0052   3
A   raid         robomimic_l